In [ ]:
# Clasificación de setas en venenosas y comestibles
# Dataset disponible y documentado en https://archive.ics.uci.edu/ml/datasets/mushroom

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [ ]:
# Carga de datos
df = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/mushroom/agaricus-lepiota.data',
  names=['toxicity', 'cap-shape', 'cap-surface', 'cap-color', 'bruises', 'odor', 'gill-attachment', 'gill-spacing', 'gill-size',
    'gill-color', 'stalk-shape', 'stalk-root', 'stalk-surface-above-ring', 'stalk-surface-below-ring', 'stalk-color-above-ring',
    'stalk-color-below-ring', 'veil-type', 'veil-color', 'ring-number', 'ring-type', 'spore-print-color', 'population', 'habitat'],
  na_values = '?'
)

In [ ]:
df.info()

In [ ]:
# Variable target: toxicity
# Valores:
#    'e': comestible (edible)
#    'p': venenosa (poisonous)

target = 'toxicity'

In [ ]:
(100*df[target].value_counts()/df.shape[0]).sort_index().plot(kind='bar')

In [ ]:
# Recodificar como números todos los atributos categóricos
for atrib in df.columns:
  if atrib != target:
    df[atrib] = LabelEncoder().fit_transform(df[atrib].values)

In [ ]:
x = df.drop([target], axis=1)
y = df[target]

In [ ]:
RANDOM_SEED = 134

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, train_size=0.8, random_state=RANDOM_SEED)

In [ ]:
clf = DecisionTreeClassifier()
clf.fit(x_train, y_train)

In [ ]:
acc_rate_train = accuracy_score(y_test, clf.predict(x_test))
print("Tasa de acierto en dataset de validación: ", acc_rate_train)

acc_rate_valid = accuracy_score(y_train, clf.predict(x_train))
print("Tasa de acierto en dataset de entrenamiento: ", acc_rate_valid)

In [ ]:
clf.classes_

In [ ]:
pd.Series(clf.feature_importances_, index=clf.feature_names_in_).nlargest(clf.feature_names_in_.shape[0]).sort_values(ascending=True).plot(kind='barh')

In [ ]:
import pydot
from IPython.display import Image
from sklearn import tree

from io import StringIO

In [ ]:
dot_data = StringIO()

tree.export_graphviz(clf, out_file = dot_data, proportion = True,
                     feature_names = x_train.columns,
                     rounded = True, filled = True)

graph = pydot.graph_from_dot_data(dot_data.getvalue())
Image(graph[0].create_png())

In [ ]:
# Basado en https://jss367.github.io/visualize-a-decision-tree-generated-by-machine-learning.html
# Código de colores dependiendo de la proporción de positivos.
# Si 100% positivos rojo (#FF0000), si 100% negativos verde (#00FF00), y entre ellos una gradación
#   de colores.
# Color rojo para positivo y verde para negativo porque "positivo" significa que la seta es venenosa.

import pydotplus

dot_data = tree.export_graphviz(clf,
                                feature_names=df.columns[1:],
                                out_file=None,
                                filled=True,
                                rounded=True,
                                special_characters=True)
graph = pydotplus.graph_from_dot_data(dot_data)
nodes = graph.get_node_list()
#colors = []
for node in nodes:
    if node.get_label():
        values = [int(float(ii)) for ii in node.get_label().split('value = [')[1].split(']')[0].split(',')]
        values = [int(255 * v / sum(values)) for v in values]
        color = '#{:02x}{:02x}00'.format(values[1], values[0])
#        colors.append(color)
        node.set_fillcolor(color)

Image(graph.create_png())
